# FEDE Vector DB — Smoke Test

This notebook exercises the full `vector_db` stack against a live Qdrant instance running in Docker.

**Prerequisites**
```bash
docker compose up -d
# wait for healthy: docker compose ps
```

All embeddings here are random unit vectors — the goal is to verify that indexing, retrieval, and hierarchical search work end-to-end, not to test semantic quality.

## 0. Setup

In [1]:
import sys, os
# Make the repo root importable when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

import random
import numpy as np

from vector_db.config import QdrantConfig
from vector_db.collections import CollectionManager
from vector_db.schemas import CollectionNames
from vector_db.indexer import ScriptIndexer, SceneRecord, SentenceRecord
from vector_db.retrieval import ScriptRetriever

VECTOR_SIZE = 64  # small dimension for fast fake embeddings
SEED = 42
rng = np.random.default_rng(SEED)

In [2]:
# Point at the Docker container — adjust host/port if needed
cfg = QdrantConfig(
    mode="server",
    host="localhost",
    port=6333,
    https=False,
    vector_size=VECTOR_SIZE,
    scripts_on_disk=False,
    hnsw_on_disk=False,
    int8_quantization=False,  # off for smoke testing
)

print(f"Config: {cfg.mode} @ {cfg.host}:{cfg.port}, vector_size={cfg.vector_size}")

Config: server @ localhost:6333, vector_size=64


## 1. Connection check

In [3]:
import time
from vector_db.client import get_qdrant_client, reset_client

reset_client()  # clear any cached singleton from a previous run

# Qdrant can take a few seconds to finish booting after container start.
# Retry with backoff so the notebook doesn't fail on a fresh container.
for attempt in range(10):
    try:
        client = get_qdrant_client(cfg)
        collections = client.get_collections()
        print(f"Connected (attempt {attempt + 1}). Existing collections:",
              [c.name for c in collections.collections])
        break
    except Exception as exc:
        print(f"Attempt {attempt + 1}/10 failed: {exc!r} — retrying in 2 s…")
        reset_client()
        time.sleep(2)
else:
    raise RuntimeError("Could not connect to Qdrant after 10 attempts. Is the container running?\n"
                       "  docker compose up -d && docker compose ps")

Connected (attempt 1). Existing collections: ['scenes', 'sentences']


## 2. Initialize collections

In [4]:
manager = CollectionManager(cfg)

# Wipe and recreate both collections for a clean slate each run
for col in CollectionNames:
    if manager.collection_exists(col):
        print(f"Resetting existing collection: {col.value}")
        manager.reset_collection(col)

manager.initialize_collections()
print("Collections after init:", manager.list_collections())

Resetting existing collection: scenes
Resetting existing collection: sentences
Collections after init: ['scenes', 'sentences']


In [5]:
# Verify both collections are empty
for col in CollectionNames:
    count = manager.get_collection_count(col)
    print(f"  {col.value}: {count} vectors")

  scenes: 0 vectors
  sentences: 0 vectors


## 3. Fake data generation

We create two fake movies, each with 5 scenes and ~4 sentences per scene.

In [6]:
def unit_vector(size: int) -> list[float]:
    """Random L2-normalised vector."""
    v = rng.standard_normal(size).astype(np.float32)
    return (v / np.linalg.norm(v)).tolist()


LINE_TYPES = ["dialogue", "dialogue", "description", "transition"]
CHARACTERS = ["ALICE", "BOB", "CHARLIE", "DETECTIVE"]

FAKE_MOVIES = [
    {"movie_id": "tt0000001", "title": "The Lost Signal"},
    {"movie_id": "tt0000002", "title": "Echo Chamber"},
]

all_scenes: list[SceneRecord] = []
all_sentences: list[SentenceRecord] = []

global_position = 0

for movie in FAKE_MOVIES:
    for scene_idx in range(5):
        scene_id = f"{movie['movie_id']}-scene-{scene_idx:02d}"
        scene_title = f"INT. LOCATION_{scene_idx} - DAY"
        scene_lines = []
        scene_chars = []

        sentences_in_scene: list[SentenceRecord] = []
        for line_idx in range(4):
            line_type = LINE_TYPES[line_idx % len(LINE_TYPES)]
            char = CHARACTERS[line_idx % len(CHARACTERS)] if line_type == "dialogue" else None
            text = (
                f"{char}: Line {line_idx} of scene {scene_idx} in '{movie['title']}'"
                if char else
                f"[{line_type.upper()}] Scene {scene_idx}, line {line_idx}"
            )
            if char:
                scene_chars.append(char)
            scene_lines.append(text)

            sentences_in_scene.append(SentenceRecord(
                movie_id=movie["movie_id"],
                movie_title=movie["title"],
                scene_id=scene_id,
                scene_index=scene_idx,
                text=text,
                line_type=line_type,
                position_in_script=global_position,
                embedding=unit_vector(VECTOR_SIZE),
                character_name=char,
            ))
            global_position += 1

        all_scenes.append(SceneRecord(
            movie_id=movie["movie_id"],
            movie_title=movie["title"],
            scene_id=scene_id,
            scene_index=scene_idx,
            text="\n".join(scene_lines),
            embedding=unit_vector(VECTOR_SIZE),
            scene_title=scene_title,
            character_names=list(set(scene_chars)),
        ))
        all_sentences.extend(sentences_in_scene)

print(f"Generated {len(all_scenes)} scenes and {len(all_sentences)} sentences across {len(FAKE_MOVIES)} movies.")

Generated 10 scenes and 40 sentences across 2 movies.


## 4. Indexing

In [7]:
indexer = ScriptIndexer(cfg)

# Index each movie's scenes and sentences in one batch call
for movie in FAKE_MOVIES:
    mid = movie["movie_id"]
    movie_scenes = [s for s in all_scenes if s.movie_id == mid]
    movie_sents = [s for s in all_sentences if s.movie_id == mid]
    indexer.index_movie_batch(movie_scenes, movie_sents)
    print(f"Indexed '{movie['title']}': {len(movie_scenes)} scenes, {len(movie_sents)} sentences")

Indexed 'The Lost Signal': 5 scenes, 20 sentences
Indexed 'Echo Chamber': 5 scenes, 20 sentences


In [8]:
# Verify counts
for col in CollectionNames:
    count = manager.get_collection_count(col)
    print(f"  {col.value}: {count} vectors")

assert manager.get_collection_count(CollectionNames.SCENES) == len(all_scenes)
assert manager.get_collection_count(CollectionNames.SENTENCES) == len(all_sentences)
print("✓ Counts match expected")

  scenes: 10 vectors
  sentences: 40 vectors
✓ Counts match expected


## 5. Idempotency — re-indexing the same data should not change counts

In [9]:
indexer.index_movie_batch(all_scenes, all_sentences)

for col in CollectionNames:
    count = manager.get_collection_count(col)
    print(f"  {col.value}: {count} vectors (after re-index)")

assert manager.get_collection_count(CollectionNames.SCENES) == len(all_scenes)
assert manager.get_collection_count(CollectionNames.SENTENCES) == len(all_sentences)
print("✓ Upsert is idempotent — no duplicates created")

  scenes: 10 vectors (after re-index)
  sentences: 40 vectors (after re-index)
✓ Upsert is idempotent — no duplicates created


## 6. Flat search — scenes

In [10]:
retriever = ScriptRetriever(cfg)
query_vec = unit_vector(VECTOR_SIZE)

scene_results = retriever.search_scenes(query_vec, top_k=5)
print(f"search_scenes returned {len(scene_results)} results:\n")
for r in scene_results:
    print(f"  [{r.score:.4f}] {r.movie_title} | {r.scene_title} | chars: {r.character_names}")

assert len(scene_results) == 5
assert all(r.movie_id in {m['movie_id'] for m in FAKE_MOVIES} for r in scene_results)
print("\n✓ search_scenes assertions passed")

search_scenes returned 5 results:

  [0.0632] Echo Chamber | INT. LOCATION_4 - DAY | chars: ['BOB', 'ALICE']
  [0.0584] Echo Chamber | INT. LOCATION_1 - DAY | chars: ['BOB', 'ALICE']
  [0.0336] Echo Chamber | INT. LOCATION_2 - DAY | chars: ['BOB', 'ALICE']
  [-0.0300] The Lost Signal | INT. LOCATION_0 - DAY | chars: ['BOB', 'ALICE']
  [-0.0890] The Lost Signal | INT. LOCATION_2 - DAY | chars: ['BOB', 'ALICE']

✓ search_scenes assertions passed


## 7. Flat search — sentences

In [11]:
sent_results = retriever.search_sentences(query_vec, top_k=8)
print(f"search_sentences returned {len(sent_results)} results:\n")
for r in sent_results:
    char_str = f" ({r.character_name})" if r.character_name else ""
    print(f"  [{r.score:.4f}] [{r.line_type}]{char_str} pos={r.position_in_script} | {r.text[:60]}")

assert len(sent_results) == 8
assert all(r.line_type in ("dialogue", "description", "transition") for r in sent_results)
print("\n✓ search_sentences assertions passed")

search_sentences returned 8 results:

  [0.3890] [description] pos=18 | [DESCRIPTION] Scene 4, line 2
  [0.3164] [description] pos=10 | [DESCRIPTION] Scene 2, line 2
  [0.2252] [transition] pos=11 | [TRANSITION] Scene 2, line 3
  [0.1797] [dialogue] (ALICE) pos=8 | ALICE: Line 0 of scene 2 in 'The Lost Signal'
  [0.1791] [transition] pos=19 | [TRANSITION] Scene 4, line 3
  [0.1408] [transition] pos=27 | [TRANSITION] Scene 1, line 3
  [0.1150] [dialogue] (ALICE) pos=32 | ALICE: Line 0 of scene 3 in 'Echo Chamber'
  [0.1138] [dialogue] (BOB) pos=37 | BOB: Line 1 of scene 4 in 'Echo Chamber'

✓ search_sentences assertions passed


## 8. movie_id filter — results restricted to one movie

In [12]:
target_movie = FAKE_MOVIES[0]

filtered_scenes = retriever.search_scenes(query_vec, top_k=10, movie_id_filter=target_movie["movie_id"])
print(f"Filtered scene search for '{target_movie['title']}': {len(filtered_scenes)} results")
assert all(r.movie_id == target_movie["movie_id"] for r in filtered_scenes), "Filter leak: wrong movie in results"

filtered_sents = retriever.search_sentences(query_vec, top_k=10, movie_id_filter=target_movie["movie_id"])
print(f"Filtered sentence search for '{target_movie['title']}': {len(filtered_sents)} results")
assert all(r.movie_id == target_movie["movie_id"] for r in filtered_sents), "Filter leak: wrong movie in results"

print("✓ movie_id_filter correctly restricts results to one movie")

Filtered scene search for 'The Lost Signal': 5 results
Filtered sentence search for 'The Lost Signal': 10 results
✓ movie_id_filter correctly restricts results to one movie


## 9. Hierarchical search

In [13]:
hier_results = retriever.hierarchical_search(query_vec, top_k=5, sentence_pool=40)
print(f"hierarchical_search returned {len(hier_results)} results:\n")
for r in hier_results:
    print(f"  [{r.score:.4f}] {r.movie_title} | {r.scene_id} | {r.scene_title}")

assert len(hier_results) <= 5
# Results must be sorted by score descending
scores = [r.score for r in hier_results]
assert scores == sorted(scores, reverse=True), "Results are not sorted by score"
# All returned scene_ids must be unique
assert len({r.scene_id for r in hier_results}) == len(hier_results), "Duplicate scenes in results"
print("\n✓ hierarchical_search assertions passed")

hierarchical_search returned 5 results:

  [0.3890] The Lost Signal | tt0000001-scene-04 | INT. LOCATION_4 - DAY
  [0.3164] The Lost Signal | tt0000001-scene-02 | INT. LOCATION_2 - DAY
  [0.1408] Echo Chamber | tt0000002-scene-01 | INT. LOCATION_1 - DAY
  [0.1150] Echo Chamber | tt0000002-scene-03 | INT. LOCATION_3 - DAY
  [0.1138] Echo Chamber | tt0000002-scene-04 | INT. LOCATION_4 - DAY

✓ hierarchical_search assertions passed


## 10. Hierarchical search — sentence-only path

We construct a query vector that is identical to one of the indexed sentence embeddings.
That sentence's parent scene should appear in the results even if the scene-level search
misses it (simulating a case where granular sentence detail drives retrieval).

In [14]:
# Pick a specific sentence and use its exact embedding as the query
target_sentence = all_sentences[0]
exact_query = target_sentence.embedding

hier_exact = retriever.hierarchical_search(exact_query, top_k=5, sentence_pool=40)
returned_scene_ids = {r.scene_id for r in hier_exact}

print(f"Query matched sentence in scene: {target_sentence.scene_id}")
print(f"Hierarchical results include that scene: {target_sentence.scene_id in returned_scene_ids}")
print("Top result:", hier_exact[0].scene_id if hier_exact else "none")

assert target_sentence.scene_id in returned_scene_ids, "Exact sentence embedding should surface its parent scene"
print("\n✓ Sentence-path scene correctly surfaced")

Query matched sentence in scene: tt0000001-scene-00
Hierarchical results include that scene: True
Top result: tt0000001-scene-00

✓ Sentence-path scene correctly surfaced


## 11. Deterministic point IDs

Re-indexing the same record must produce the same point ID (uuid5 stability).

In [15]:
from vector_db.indexer import _scene_point_id, _sentence_point_id

scene = all_scenes[0]
pid1 = indexer.index_scene(scene)
pid2 = indexer.index_scene(scene)
assert pid1 == pid2, "Point ID changed between identical index calls"
print(f"Scene point ID (stable): {pid1}")

sent = all_sentences[0]
spid1 = indexer.index_sentence(sent)
spid2 = indexer.index_sentence(sent)
assert spid1 == spid2
print(f"Sentence point ID (stable): {spid1}")

# IDs are different between scenes and sentences with same movie/scene
assert _scene_point_id(scene.movie_id, scene.scene_id) != _sentence_point_id(scene.movie_id, scene.scene_id, 0)
print("✓ IDs are stable and scene/sentence namespaces do not collide")

Scene point ID (stable): 1ad1ee1e-3039-5290-b254-674322c59364
Sentence point ID (stable): 1ceb836d-0ca1-58ff-941e-b24bb8d4b7a6
✓ IDs are stable and scene/sentence namespaces do not collide


## 12. Summary

In [16]:
print("=" * 55)
print("FEDE Vector DB smoke test complete")
print("=" * 55)
for col in CollectionNames:
    print(f"  {col.value:12s}: {manager.get_collection_count(col)} vectors")
print()
print("All assertions passed ✓")

FEDE Vector DB smoke test complete
  scenes      : 10 vectors
  sentences   : 40 vectors

All assertions passed ✓
